# 卫星嵌入

## 介绍

传统遥感工作流将每个分析任务视为孤立问题，每个新项目都需要单独的标注数据、模型训练和验证。这种逐任务方法成本高、速度慢且难以扩展。

卫星嵌入提供了一种根本不同的方法。在大量卫星图像上预训练的基础模型将每个图像块或像素压缩成紧凑的数值向量（嵌入），捕获光谱特征、空间纹理和土地覆盖特征等语义内容。一旦计算完成，嵌入可作为通用表示，无需重新训练即可在多个任务中复用。

卫星嵌入数据集生态系统迅速发展。像[Clay](https://clay-foundation.github.io/model)、Major TOM、TESSERA和AlphaEarth等项目已发布覆盖全球的预计算嵌入，分辨率从10米到5公里不等。本教程使用`geoai` Python包介绍卫星嵌入工作流，涵盖三个系统：用于基于块的嵌入的Clay Foundation Model、用于基于像素的时间嵌入的TESSERA，以及通过Google Earth Engine进行云分析的AlphaEarth。

## 学习目标

在本教程结束时，您将能够：

- 解释什么是卫星嵌入，并区分基于块和基于像素的格式
- 使用`geoai`浏览和查询可用嵌入数据集的注册表
- 从Hugging Face下载和加载Clay Foundation Model嵌入
- 使用主成分分析（PCA）可视化高维嵌入空间
- 在没有标注数据的情况下对嵌入进行聚类以发现空间模式
- 执行相似性搜索以找到具有匹配特征的位置
- 在嵌入特征上训练轻量级分类器（k-NN、随机森林、逻辑回归）
- 跨位置比较嵌入以进行变化检测
- 下载、可视化和采样TESSERA时间嵌入
- 在Google Earth Engine中探索AlphaEarth卫星嵌入以进行多年比较

## 什么是卫星嵌入？

卫星嵌入是一个固定长度的数值向量，编码卫星图像块或像素的内容。基础模型通过深度神经网络处理原始图像来生成这些向量，创建一个高维空间，其中几何接近度编码语义相似度。

卫星嵌入有两种格式。**基于块的嵌入**将整个图像瓦片总结为存储在GeoParquet文件中的单个向量，适合检索和分类等场景级任务。**基于像素的嵌入**为GeoTIFF栅格中的每个像素分配一个向量，支持分割和逐像素分类等细粒度分析。

`geoai`包提供了来自多个基础模型的嵌入数据集统一注册表，允许您列出、过滤和检索详细的元数据。

## 设置环境

安装所需的包并导入本教程中使用的库。

In [ ]:
# %pip install "geoai-py[extra]"

In [ ]:
import geoai
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

## 浏览可用的嵌入数据集

`geoai`包提供了由[TorchGeo](https://torchgeo.readthedocs.io)数据集类支持的嵌入数据集注册表。使用`list_embedding_datasets()`查看可用的数据集。

In [ ]:
df = geoai.list_embedding_datasets(verbose=False)
df

| 名称              | 类别                     | 类型  | 空间范围       | 分辨率       | 时间范围         | 维度       |
| ----------------- | ------------------------ | ----- | -------------- | ------------ | --------------- | ---------- |
| clay              | ClayEmbeddings           | patch | Global\*       | 5.12 km      | 2018-2023\*     | 768        |
| major_tom         | MajorTOMEmbeddings       | patch | Global         | 2.14-3.56 km | 2015-2024\*     | 2048       |
| earth_index       | EarthIndexEmbeddings     | 块    | 全球           | 320米        | 2024            | 384        |
| earth_embeddings  | EarthEmbeddings          | patch | Global\*       | 2.24-3.84 km | 2015-2024\*     | 256-1152   |
| copernicus_embed  | CopernicusEmbed          | 像素  | 全球           | 0.25度       | 2021            | 768        |
| presto            | PrestoEmbeddings         | 像素  | 多哥           | 10米         | 2019-2020       | 128        |
| tessera           | TesseraEmbeddings        | pixel | Global         | 10 m         | 2017-2024\*     | 128        |
| google_satellite  | GoogleSatelliteEmbedding | 像素  | 全球           | 10米         | 2017-2024       | 64         |
| embedded_seamless | EmbeddedSeamlessData     | 像素  | 全球           | 30米         | 2000-2024       | 12         |

您可以按类型过滤，只查看基于块或基于像素的数据集。

In [ ]:
geoai.list_embedding_datasets(kind="patch", verbose=False)

In [ ]:
geoai.list_embedding_datasets(kind="pixel", verbose=False)

使用`get_embedding_info()`详细检查特定数据集。

In [ ]:
info = geoai.get_embedding_info("clay")
for key, value in info.items():
    print(f"{key}: {value}")

```text
class_name: ClayEmbeddings
kind: patch
base: NonGeoDataset
spatial_extent: Global*
spatial_resolution: 5.12 km
temporal_extent: 2018-2023*
dimensions: 768
dtype: float32
license: ODC-By-1.0
description: Clay Foundation Model的Clay v0和v1.5嵌入。存储为GeoParquet文件。
paper: https://clay-foundation.github.io/model/
data_source: https://source.coop/clay/clay-model-v0-embeddings
```

## 探索基于块的嵌入

基于块的嵌入将整个图像瓦片压缩成单个向量。Clay Foundation Model从Sentinel-2、Landsat、NAIP、Sentinel-1和MODIS数据等来源生成768维嵌入。

我们将使用[Hugging Face](https://huggingface.co/datasets/made-with-clay/classify-embeddings-sf-baseball-marinas)上旧金山湾区的Clay嵌入，包含来自20个瓦片的NAIP图像的768维嵌入，以及棒球场（类别0）和码头（类别1）的标注位置。

### 下载Clay嵌入

列出Hugging Face仓库中可用的GeoParquet文件。

In [ ]:
repo_id = "made-with-clay/classify-embeddings-sf-baseball-marinas"

api = HfApi()
embedding_files = [
    f.path
    for f in api.list_repo_tree(repo_id, repo_type="dataset")
    if f.path.endswith(".gpq")
]
print(f"Found {len(embedding_files)} embedding tiles")

```text
找到20个嵌入瓦片
```

下载所有瓦片并将它们连接成单个GeoDataFrame。

In [ ]:
all_gdfs = []
for f in embedding_files:
    path = hf_hub_download(repo_id, f, repo_type="dataset")
    gdf = gpd.read_parquet(path)
    all_gdfs.append(gdf)

embeddings_gdf = pd.concat(all_gdfs, ignore_index=True)
embeddings_gdf = gpd.GeoDataFrame(
    embeddings_gdf, geometry="geometry", crs=all_gdfs[0].crs
)
print(f"Combined: {len(embeddings_gdf)} patches")
print(f"Bounds: {embeddings_gdf.total_bounds}")
print(f"Embedding dimension: {len(embeddings_gdf.iloc[0]['embeddings'])}")

```text
合并后：36024个块
边界：[-122.62763615   37.56033178 -122.3108981    37.87715656]
嵌入维度：768
```

下载两个类别的标注点位置：棒球场（类别0）和码头（类别1）。

In [ ]:
labels_file = hf_hub_download(repo_id, "baseball.geojson", repo_type="dataset")
labels_gdf = gpd.read_file(labels_file)
print(f"Labeled locations: {len(labels_gdf)}")
print(f"Class distribution:")
print(labels_gdf["class"].value_counts())

```text
标注位置：97
类别分布：
class
0    74
1    23
Name: count, dtype: int64
```

### 提取嵌入向量

将嵌入向量堆叠成NumPy矩阵并提取质心坐标用于地理绘图。

In [ ]:
embeddings = np.stack(embeddings_gdf["embeddings"].values)
centroids = embeddings_gdf.geometry.centroid
coords_x = centroids.x.values
coords_y = centroids.y.values

print(f"Embeddings shape: {embeddings.shape}")
print(f"X range: [{coords_x.min():.4f}, {coords_x.max():.4f}]")
print(f"Y range: [{coords_y.min():.4f}, {coords_y.max():.4f}]")

```text
嵌入形状：(36024, 768)
X范围：[-122.6268, -122.3118]
Y范围：[37.5610, 37.8765]
```

## 可视化嵌入

PCA（主成分分析）将高维嵌入投影到二维空间，同时保留重要的结构关系。绘制几个单独的嵌入向量以查看它们在768维上的原始模式。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for i, ax in enumerate(axes):
    idx = i * (len(embeddings) // 3)
    ax.plot(embeddings[idx], linewidth=0.5)
    ax.set_title(f"Patch {idx}")
    ax.set_xlabel("Dimension")
    ax.set_ylabel("Value")
plt.tight_layout()
plt.show()

将所有嵌入投影到二维PCA散点图中。

In [ ]:
fig = geoai.visualize_embeddings(
    embeddings,
    method="pca",
    figsize=(10, 8),
    s=3,
    alpha=0.4,
    title="PCA of Clay Embeddings (SF Bay Area)",
)
plt.show()

靠近的点代表具有相似视觉和光谱特征的瓦片，不同的聚类通常对应不同的土地覆盖类型。

## 嵌入聚类

聚类根据嵌入空间中的接近度将嵌入分组为离散类别，无需标注数据。`cluster_embeddings()`函数支持K-means聚类。

In [ ]:
result = geoai.cluster_embeddings(embeddings, n_clusters=8, method="kmeans")
cluster_labels = result["labels"]
print(f"Number of clusters: {result['n_clusters']}")
print(f"Cluster sizes: {np.bincount(cluster_labels)}")

```text
聚类数量：8
聚类大小：[4284 5424 7097 5556 4136 3952 2936 2639]
```

通过用聚类分配给PCA嵌入图着色来可视化聚类。

In [ ]:
fig = geoai.visualize_embeddings(
    embeddings,
    labels=cluster_labels,
    method="pca",
    figsize=(10, 8),
    s=5,
    alpha=0.5,
    title="K-Means Clusters of Clay Embeddings",
)
plt.show()

在地理上映射聚类分配以查看它们如何对应空间模式。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
n_clusters = len(set(cluster_labels))
scatter = ax.scatter(
    coords_x,
    coords_y,
    c=cluster_labels,
    cmap=plt.colormaps["tab10"].resampled(n_clusters),
    vmin=-0.5,
    vmax=n_clusters - 0.5,
    s=3,
    alpha=0.6,
)
plt.colorbar(scatter, ax=ax, label="Cluster", ticks=range(n_clusters))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Geographic Distribution of Embedding Clusters")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

检查聚类的地理分布可以揭示空间模式，如城市区域与植被区域，这些模式无需人为定义类别就能自然出现。

## 相似性搜索

相似性搜索找到嵌入与查询最相似的位置，适用于灾害响应、生态监测和城市规划。`embedding_similarity()`函数计算查询与所有其他嵌入之间的成对余弦相似度分数。

In [ ]:
query_idx = 0
query = embeddings[query_idx]
print(
    f"Query location (lat, lon): ({coords_y[query_idx]:.4f}, {coords_x[query_idx]:.4f})"
)

results = geoai.embedding_similarity(
    query=query, embeddings=embeddings, metric="cosine", top_k=10
)

print("\nTop 10 most similar locations:")
for rank, (idx, score) in enumerate(
    zip(results["indices"], results["scores"]), start=1
):
    print(
        f"  {rank}. Index {idx}: similarity={score:.4f}, "
        f"location=({coords_y[idx]:.4f}, {coords_x[idx]:.4f})"
    )

```text
查询位置（纬度，经度）：(37.8764, -122.5639)

前10个最相似的位置：
  1. 索引0：相似度=1.0000，位置=(37.8764, -122.5639)
  2. 索引1822：相似度=0.9521，位置=(37.8761, -122.5636)
  3. 索引1935：相似度=0.8796，位置=(37.8720, -122.5653)
  4. 索引1860：相似度=0.8707，位置=(37.8747, -122.5636)
  5. 索引38：相似度=0.8695，位置=(37.8750, -122.5639)
  6. 索引1892：相似度=0.8582，位置=(37.8734, -122.5741)
  7. 索引1820：相似度=0.8579，位置=(37.8761, -122.5671)
  8. 索引450：相似度=0.8561，位置=(37.8609, -122.5081)
  9. 索引606：相似度=0.8520，位置=(37.8554, -122.5011)
  10. 索引1852：相似度=0.8385，位置=(37.8748, -122.5775)
```

在地理图上可视化查询点及其最近邻。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Background: all embeddings in gray
ax.scatter(coords_x, coords_y, c="lightgray", s=1, alpha=0.3)

# Highlight nearest neighbors
nn_indices = results["indices"]
ax.scatter(
    coords_x[nn_indices],
    coords_y[nn_indices],
    c="blue",
    s=50,
    marker="o",
    label="Nearest Neighbors",
    edgecolors="black",
    linewidths=0.5,
)

# Highlight the query point
ax.scatter(
    coords_x[query_idx],
    coords_y[query_idx],
    c="red",
    s=100,
    marker="*",
    label="Query",
    edgecolors="black",
    linewidths=0.5,
    zorder=5,
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Similarity Search: Query and Nearest Neighbors")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

分数为1.0表示相同的嵌入，而较低的分数表示相似度降低。地理图揭示相似嵌入是否在空间上聚类。

## 在嵌入上训练分类器

因为基础模型已经提取了丰富的特征，只需要一个简单的分类器。工作流包括通过空间连接将标注点与嵌入块匹配，然后分割成训练集和验证集。

In [ ]:
# Ensure both GeoDataFrames use the same CRS
if labels_gdf.crs != embeddings_gdf.crs:
    labels_gdf = labels_gdf.to_crs(embeddings_gdf.crs)

# Spatial join: find which embedding patch each labeled point falls within
joined = gpd.sjoin(labels_gdf, embeddings_gdf, how="inner", predicate="within")
print(f"Matched {len(joined)} labeled points to embedding patches")
print(f"Class distribution: {joined['class'].value_counts().to_dict()}")

```text
将106个标注点匹配到嵌入块
类别分布：{0: 79, 1: 27}
```

提取每个匹配点的嵌入向量。

In [ ]:
labeled_embeddings = np.stack(
    [embeddings_gdf.iloc[idx]["embeddings"] for idx in joined["index_right"]]
)
class_labels = joined["class"].values

print(f"Labeled embeddings shape: {labeled_embeddings.shape}")
print(f"Labels shape: {class_labels.shape}")

```text
标注嵌入形状：(106, 768)
标签形状：(106,)
```

使用分层抽样分成70%训练和30%验证。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    labeled_embeddings,
    class_labels,
    test_size=0.3,
    random_state=42,
    stratify=class_labels,
)
print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")

```text
训练：74个样本
验证：32个样本
```

训练k-NN分类器。

In [ ]:
label_names = ["Baseball Field", "Marina"]

result = geoai.train_embedding_classifier(
    train_embeddings=X_train,
    train_labels=y_train,
    val_embeddings=X_val,
    val_labels=y_val,
    method="knn",
    n_neighbors=5,
    label_names=label_names,
)

print(f"\nTrain accuracy: {result['train_accuracy']:.2%}")
print(f"Val accuracy:   {result['val_accuracy']:.2%}")

```text
precision    recall  f1-score   support

Baseball Field      0.864     0.792     0.826        24
        Marina      0.500     0.625     0.556         8

      accuracy                          0.750        32
     macro avg      0.682     0.708     0.691        32
  weighted avg      0.773     0.750     0.758        32

训练准确率：86.49%
验证准确率：75.00%
```

比较三种分类器方法。

In [ ]:
methods = ["knn", "random_forest", "logistic_regression"]
results_summary = []

for method in methods:
    res = geoai.train_embedding_classifier(
        train_embeddings=X_train,
        train_labels=y_train,
        val_embeddings=X_val,
        val_labels=y_val,
        method=method,
        label_names=label_names,
        verbose=False,
    )
    results_summary.append(
        {
            "Method": method,
            "Train Acc": f"{res['train_accuracy']:.2%}",
            "Val Acc": f"{res['val_accuracy']:.2%}",
        }
    )

pd.DataFrame(results_summary)

在PCA空间中可视化标注嵌入以查看两个类别分离得有多好。

In [ ]:
fig = geoai.visualize_embeddings(
    labeled_embeddings,
    labels=class_labels,
    label_names=label_names,
    method="pca",
    figsize=(8, 8),
    s=30,
    alpha=0.8,
    title="PCA of Labeled Embeddings (Baseball vs Marina)",
)
plt.show()

基础模型完成特征提取的繁重工作，下游分类器在几秒钟内用很少的标注样本完成训练。

## 比较嵌入进行变化检测

比较同一位置不同时间的嵌入可以揭示景观变化。`compare_embeddings()`函数量化相似度，其中较高的余弦值表示稳定性，较低的值表示变化。

在这里，我们通过比较不同空间块的嵌入来演示。

In [ ]:
n = len(embeddings)
half = n // 2
emb_a = embeddings[:half]
emb_b = embeddings[half : half + half]

similarity = geoai.compare_embeddings(emb_a, emb_b, metric="cosine")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(similarity, bins=50, edgecolor="black", alpha=0.7)
ax.axvline(
    similarity.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {similarity.mean():.3f}",
)
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Count")
ax.set_title("Embedding Similarity Distribution")
ax.legend()
plt.tight_layout()
plt.show()

在真正的时间比较中，您会比较相同位置不同日期的嵌入，低相似度的异常值将表示发生重大变化的区域。

## 使用TorchGeo加载数据集

对于高级用法，通过`geoai.load_embedding_dataset()`直接加载TorchGeo数据集类，以访问变换、采样和内置绘图功能。

请注意，TorchGeo的`ClayEmbeddings`类期望GeoParquet文件中有`date`或`datetime`列。某些文件可能不包含此列，需要回退到geopandas。

In [ ]:
single_file = hf_hub_download(repo_id, embedding_files[0], repo_type="dataset")
ds = geoai.load_embedding_dataset("clay", root=single_file)

print(f"Dataset length: {len(ds)}")
print(f"Dataset type: {type(ds).__name__}")

```text
数据集长度：1786
数据集类型：ClayEmbeddings
```

In [ ]:
try:
    sample = ds[0]
    print(f"Sample keys: {list(sample.keys())}")
    print(f"Embedding shape: {sample['embedding'].shape}")

    fig = ds.plot(sample)
    plt.show()
except KeyError as e:
    print(
        f"Note: This parquet file is missing the '{e.args[0]}' column "
        f"expected by TorchGeo's ClayEmbeddings class."
    )
    print("For such files, use geopandas directly (as shown above).")
    print("The TorchGeo class works best with official Clay data products.")

## 使用TESSERA时间嵌入

[TESSERA](https://github.com/ucam-eo/tessera)（地球表面光谱时间嵌入表示与分析）从Sentinel-1和Sentinel-2时间序列图像生成全球10米分辨率的128通道表示图。`geoai`包包含用于检查可用性、下载、可视化和采样TESSERA嵌入的函数。

In [ ]:
# %pip install "geoai-py[extra]" geotessera

### 检查数据可用性

检查可用的年份以及覆盖您区域的瓦片数量。

In [ ]:
years = geoai.tessera_available_years()
print(f"Available years: {years}")

```text
可用年份：[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
```

为英国剑桥定义边界框并检查瓦片数量。

In [ ]:
bbox = (0.05, 52.15, 0.25, 52.25)
count = geoai.tessera_tile_count(bbox=bbox, year=2024)
print(f"{count} tiles available for the specified region")

```text
指定区域有6个可用瓦片
```

生成覆盖图。

In [ ]:
geoai.tessera_coverage(year=2024, output_path="tessera_coverage_2024.png")

In [ ]:
geoai.tessera_coverage(
    year=2024, region_bbox=(-10, 35, 40, 60), output_path="tessera_coverage_europe.png"
)

### 下载嵌入

TESSERA嵌入可以通过边界框、点坐标或区域文件以GeoTIFF或NumPy格式下载。

In [ ]:
bbox = (0.05, 52.15, 0.25, 52.25)

cambridge_files = geoai.tessera_download(
    bbox=bbox, year=2024, output_dir="./tessera_cambridge", output_format="tiff"
)
print(f"Downloaded {len(cambridge_files)} files")
for f in cambridge_files:
    print(f"  {f}")

通过点坐标下载单个瓦片。

In [ ]:
files = geoai.tessera_download(
    lon=0.15, lat=52.05, year=2024, output_dir="./tessera_single_tile"
)
print(f"Downloaded {len(files)} file(s)")

仅下载特定波段以减小文件大小。

In [ ]:
files = geoai.tessera_download(
    bbox=bbox, year=2024, bands=[0, 1, 2], output_dir="./tessera_rgb_only"
)
print(f"Downloaded {len(files)} files with 3 bands each")

### 可视化TESSERA嵌入

TESSERA嵌入可以通过选择128个通道中的三个渲染为假彩色RGB合成图。颜色相似的区域具有相似的时间动态。

In [ ]:
geoai.tessera_visualize_rgb(
    str(cambridge_files[0]), bands=(0, 1, 2), title="Cambridge - TESSERA Bands 0, 1, 2"
)

尝试不同的波段组合以突出时间信号的其他方面。

In [ ]:
geoai.tessera_visualize_rgb(
    str(cambridge_files[0]),
    bands=(30, 60, 90),
    title="Cambridge - TESSERA Bands 30, 60, 90",
)

不同的波段组合可以揭示季节性植被模式、城市热信号或水体动态。

### 将嵌入加载到内存

使用`tessera_fetch_embeddings()`直接将嵌入数组加载到内存中，无需将文件保存到磁盘。

In [ ]:
tiles = geoai.tessera_fetch_embeddings(bbox=(0.05, 52.15, 0.25, 52.25), year=2024)

for tile in tiles:
    print(f"Tile ({tile['lon']:.2f}, {tile['lat']:.2f}):")
    print(f"  Shape: {tile['embedding'].shape}")
    print(f"  CRS: {tile['crs']}")
    print(f"  Mean: {tile['embedding'].mean():.4f}")
    print(f"  Std: {tile['embedding'].std():.4f}")

### 在点位置采样嵌入

使用`tessera_sample_points()`在特定点提取嵌入值，它将128列（`tessera_0`到`tessera_127`）附加到输入GeoDataFrame。

In [ ]:
from shapely.geometry import Point

points = gpd.GeoDataFrame(
    {"name": ["Point A", "Point B", "Point C"]},
    geometry=[Point(0.12, 52.20), Point(0.15, 52.18), Point(0.20, 52.22)],
    crs="EPSG:4326",
)

result = geoai.tessera_sample_points(points, year=2024)

print(f"Result shape: {result.shape}")
print(
    f"\nOriginal columns: {[c for c in result.columns if not c.startswith('tessera_')]}"
)
print(f"Embedding columns: tessera_0 through tessera_127")
print(f"\nFirst few embedding values for Point A:")
print(result.iloc[0][[f"tessera_{i}" for i in range(5)]])

```text
结果形状：(3, 130)

原始列：['name', 'geometry']
嵌入列：tessera_0到tessera_127

点A的前几个嵌入值：
tessera_0    8.032325
tessera_1   -0.316233
tessera_2    0.505973
tessera_3    2.593113
tessera_4    0.442727
Name: 0, dtype: object
```

### 替代下载格式

以NumPy数组和JSON元数据文件格式下载。

In [ ]:
import json

files = geoai.tessera_download(
    bbox=bbox, year=2024, output_dir="./tessera_numpy", output_format="npy"
)

with open("./tessera_numpy/metadata.json") as f:
    meta = json.load(f)

print(f"Year: {meta['year']}")
print(f"Number of tiles: {len(meta['tiles'])}")
for tile_info in meta["tiles"]:
    arr = np.load(f"./tessera_numpy/{tile_info['file']}")
    print(f"  {tile_info['file']}: shape={arr.shape}, dtype={arr.dtype}")

使用GeoJSON或Shapefile定义下载区域。

In [ ]:
from shapely.geometry import box

region = gpd.GeoDataFrame(geometry=[box(0.05, 52.15, 0.25, 52.25)], crs="EPSG:4326")
region.to_file("cambridge_region.geojson", driver="GeoJSON")

files = geoai.tessera_download(
    region_file="cambridge_region.geojson",
    year=2024,
    output_dir="./tessera_from_region",
)
print(f"Downloaded {len(files)} files")

## 探索AlphaEarth卫星嵌入

[AlphaEarth](https://deepmind.google/discover/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail)来自Google DeepMind，包含2017年至2024年的年度卫星嵌入，分辨率为10米，可在Google Earth Engine上获取。通过跨年份比较嵌入，您可以识别景观变化而无需在本地下载任何数据。

本节需要Google Earth Engine账户。如果需要，请在<https://earthengine.google.com>创建。

In [ ]:
# %pip install -U geemap

In [ ]:
import ee
import geoai

使用您的项目进行身份验证和初始化。将`"your-ee-project"`替换为您自己的项目ID。

In [ ]:
ee.Authenticate()
ee.Initialize(project="your-ee-project")

### 使用AlphaEarth GUI的交互式地图

`geoai`包包含内置的AlphaEarth GUI小部件用于交互式探索。点击**Apply**更新可视化。

In [ ]:
m = geoai.Map(projection="globe", sidebar_visible=True)
m.add_basemap("USGS.Imagery")
m.add_alphaearth_gui()
m

### 可视化嵌入 as RGB Composites

加载AlphaEarth ImageCollection并按日期过滤以比较2017年和2024年的嵌入。

In [ ]:
m = geoai.Map(projection="globe", sidebar_visible=True)
m.add_basemap("USGS.Imagery")

lon = -121.8036
lat = 39.0372
m.set_center(lon, lat, zoom=12)
m

In [ ]:
point = ee.Geometry.Point(lon, lat)
dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

image1 = dataset.filterDate("2017-01-01", "2018-01-01").filterBounds(point).first()
image2 = dataset.filterDate("2024-01-01", "2025-01-01").filterBounds(point).first()

使用波段A01、A16和A09将两年可视化为假彩色RGB合成图。

In [ ]:
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}
m.add_ee_layer(image1, vis_params, name="2017 embeddings")
m.add_ee_layer(image2, vis_params, name="2024 embeddings")

### 通过嵌入相似性进行变化检测

计算2017年和2024年嵌入之间的点积以得到像素级相似性图。高值（接近1）表示稳定性；低值表示显著变化。

In [ ]:
dot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())

添加相似性图层，使用白色到黑色的调色板，其中较亮的像素表示更多变化。

In [ ]:
vis_params = {"min": 0, "max": 1, "palette": ["#ffffff", "#000000"]}
m.add_ee_layer(dot_prod, vis_params, name="Similarity")
m.add_colorbar(cmap="gray", label="Similarity")
m

这种基于云的方法可以扩展到AlphaEarth数据集覆盖的地球上任何区域，无需下载栅格数据。

## 关键要点

1. 卫星嵌入是来自基础模型的紧凑向量，支持相似性搜索、聚类、分类和变化检测，无需特定任务训练。
2. 基于块的嵌入（GeoParquet）总结瓦片用于场景级分析，而基于像素的嵌入（GeoTIFF）支持细粒度的逐像素工作。
3. `geoai`包提供了来自多个基础模型的嵌入数据集统一注册表，可通过一致的API访问。
4. PCA可视化揭示嵌入空间中的结构，聚类点代表具有相似特征的位置。
5. 余弦相似性搜索支持快速地理检索具有匹配特征的位置。
6. 无监督聚类无需标注数据即可揭示土地覆盖类型等自然分组。
7. 在嵌入上训练的轻量级分类器用很少的标注样本就能达到很高的准确率。
8. 跨时间或空间比较嵌入提供高效的变化检测，无需标注变化示例。
9. TESSERA时间嵌入以10米分辨率捕获全球季节性动态，支持多种下载格式和访问模式。
10. Google Earth Engine中的AlphaEarth嵌入支持基于云的全球范围多年比较，无需本地数据下载。